<a href="https://colab.research.google.com/github/RonakkudalAI/Deep-Neural-Network/blob/main/04_marketing_text_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Summarization using Hugging Face

## Marketing case study: summarizing campaign performance reports

Marketing teams produce long campaign reports, survey summaries, social media insights, and launch notes. Managers often need the key points quickly.

In this notebook, we will build an end-to-end text summarization workflow using Hugging Face.

The business goal is to turn long marketing text into short, useful summaries that help teams make faster decisions.


## Learning roadmap

1. Understand the marketing reporting problem.
2. Create realistic campaign reports.
3. Understand text summarization.
4. Learn extractive versus abstractive summarization.
5. Understand why encoder-decoder models are useful.
6. Load a Hugging Face summarization model.
7. Summarize one report.
8. Summarize multiple reports.
9. Convert summaries into business insights.
10. Discuss limitations, hallucination risk, and alternatives.


## Code microscope: important summarization words

| Term | Simple meaning | Why it matters |
|---|---|---|
| `pipeline("summarization")` | A ready-to-use Hugging Face summarization workflow. | It handles tokenization, model generation, and decoding. |
| DistilBART | A smaller BART-style encoder-decoder model. | Useful for classroom summarization because it is lighter than larger models. |
| Encoder-decoder | One part reads the input, another part generates output. | Summarization needs both understanding and writing. |
| `max_length` | Maximum length of the generated summary in tokens. | Controls how long the summary can be. |
| `min_length` | Minimum length of the generated summary in tokens. | Prevents extremely short summaries. |
| `do_sample=False` | Uses deterministic generation instead of random sampling. | Good for repeatable classroom outputs. |
| Hallucination | A generated statement not supported by the source text. | Marketing summaries must be reviewed before business use. |


## 1. Why summarization matters in marketing

Marketing teams may read:

- Campaign performance reports
- Customer survey responses
- Social media comments
- Product launch updates
- Competitor analysis notes

Summarization helps teams quickly understand what happened, what worked, what failed, and what to do next.


## 2. Simple explanation

Text summarization means making a long piece of text shorter while keeping the most important meaning.

Real-world analogy:

Imagine a marketing analyst reads a ten-page campaign report and gives the manager a five-line brief. A summarization model tries to do something similar.


## 3. Tiny example

Long text:

"The email campaign had a high open rate but a low click-through rate. Paid ads generated many impressions but few conversions. Organic social posts performed well because customers shared the product videos."

Short summary:

"Email and organic social performed well, but paid ads had weak conversions."


## 4. Extractive versus abstractive summarization

### Extractive summarization

The summary is made by selecting important sentences from the original text.

### Abstractive summarization

The model writes a new shorter version in its own words.

Hugging Face summarization pipelines usually use abstractive models such as BART, T5, or PEGASUS.

This is powerful, but it also creates risk: the model may generate a sentence that sounds correct but is not fully supported by the original text. This is called hallucination.


## 5. Why not use plain BERT for summarization?

BERT is mainly an encoder model. It is excellent at understanding text.

Summarization usually needs both:

- Understanding the input text.
- Generating a new output summary.

Encoder-decoder models are designed for this.

Examples:

- BART
- T5
- PEGASUS

In this notebook, we will use a BART-style summarization model through Hugging Face.


In [ ]:
# This cell installs the libraries used in this notebook.
# Run it once in a fresh environment.
%pip install -q transformers pandas numpy torch


In [ ]:
# pandas helps us store campaign reports and summaries in tables.
import pandas as pd

# numpy helps with numerical operations.
import numpy as np

# pipeline gives us an easy interface to pretrained Hugging Face models.
from transformers import pipeline


## 6. Create realistic marketing reports

Each report describes one marketing campaign.

Columns:

- `campaign_id`: campaign number.
- `campaign_name`: campaign title.
- `channel_focus`: main marketing channel.
- `report_text`: long campaign report.


In [ ]:
# We create a small set of marketing campaign reports.
marketing_reports = [
    {
        "campaign_id": 1,
        "campaign_name": "Spring Product Launch",
        "channel_focus": "Email and Social",
        "report_text": "The Spring Product Launch campaign ran for four weeks across email, LinkedIn, Instagram, and paid search. Email open rates were higher than the previous launch, mainly because the subject lines clearly mentioned the new product benefit. However, click-through rates were only slightly above average because the email body had too many links and no single primary call to action. LinkedIn posts generated strong engagement from existing customers, especially posts that included short product demonstration videos. Instagram reached a younger audience but produced fewer qualified leads. Paid search generated traffic quickly, but conversion quality was mixed because several keywords attracted visitors who were still in early research mode. The strongest result came from existing customer referrals after the customer success team shared launch material with account champions.",
    },
    {
        "campaign_id": 2,
        "campaign_name": "Enterprise Webinar Series",
        "channel_focus": "Webinars",
        "report_text": "The Enterprise Webinar Series targeted senior sales and revenue leaders in mid-market and enterprise companies. Registrations were strong for the first webinar because the topic focused on revenue forecasting challenges. Attendance dropped in the second webinar because reminder emails were sent too late and the session time conflicted with a major industry event. The Q&A sections were highly valuable, with many attendees asking about integrations, implementation timelines, and security reviews. Follow-up emails that included the webinar recording performed better than generic thank-you emails. Sales accepted 38 percent of webinar leads as qualified opportunities. The campaign showed that educational content works well when it is connected to urgent business problems.",
    },
    {
        "campaign_id": 3,
        "campaign_name": "Customer Advocacy Push",
        "channel_focus": "Customer Stories",
        "report_text": "The Customer Advocacy Push focused on publishing customer stories, short quotes, and peer recommendation posts. The campaign performed best on LinkedIn, where posts from customer leaders received more engagement than posts from the company page. Website traffic to the case study section increased, and visitors who read at least one customer story spent more time on product pages. The main weakness was production speed because approvals from customer legal teams took longer than expected. Sales teams reported that customer proof helped in late-stage deals, especially when prospects were comparing vendors. The campaign should continue, but the team needs a faster approval process and a larger library of industry-specific stories.",
    },
    {
        "campaign_id": 4,
        "campaign_name": "Paid Search Optimization",
        "channel_focus": "Paid Search",
        "report_text": "The Paid Search Optimization campaign tested new ad copy, landing pages, and keyword groups. Cost per click increased in competitive categories, but conversion rate improved on landing pages that used shorter forms and clearer product screenshots. Broad match keywords produced high traffic but weak lead quality. Exact match keywords produced fewer visitors but better sales conversations. The highest-performing landing page included a customer quote, a short product video, and a single demo request button. The lowest-performing page used generic messaging and asked for too many details in the form. The recommendation is to reduce broad match spending and shift budget to high-intent exact match terms.",
    },
]

# We convert the reports into a DataFrame.
reports_df = pd.DataFrame(marketing_reports)

# We display the campaign reports table.
reports_df


## 7. Explore report length

Before summarization, we check how long the reports are. Very long text may need chunking because transformer models have maximum input lengths.


In [ ]:
# We calculate word count for every report.
reports_df["word_count"] = reports_df["report_text"].str.split().str.len()

# We show campaign names and word counts.
reports_df[["campaign_name", "word_count"]]


## 8. Load a summarization model

We use a Hugging Face summarization pipeline.

The model will:

1. Read the campaign report.
2. Generate a shorter summary.
3. Return the summary text.

For classroom use, the first model download may take a few minutes.


In [ ]:
# pipeline is Hugging Face's high-level interface for common NLP tasks.
# task="summarization" tells Hugging Face to load a model suitable for generating summaries.
summarizer = pipeline(
    # This task name selects the summarization workflow.
    task="summarization",

    # This checkpoint points to a DistilBART model trained for news-style summarization.
    # DistilBART is smaller than full BART, so it is more practical for classroom demonstrations.
    model="sshleifer/distilbart-cnn-12-6",
)


## 9. Summarize one report

We start with one campaign report so the output is easy to inspect.


In [ ]:
# We select the first report from the DataFrame.
sample_report = reports_df.loc[0, "report_text"]

# We pass the report into the summarization pipeline.
sample_summary_output = summarizer(
    # The first argument is the long text we want to summarize.
    sample_report,

    # max_length is the maximum generated summary length in tokens.
    # A token can be a word or part of a word.
    max_length=80,

    # min_length prevents the model from producing an extremely short summary.
    min_length=30,

    # do_sample=False makes generation more deterministic.
    # This is helpful in classrooms because students get more repeatable outputs.
    do_sample=False,
)

# The pipeline returns a list of dictionaries.
# For one input report, we take the first dictionary and read the "summary_text" field.
sample_summary = sample_summary_output[0]["summary_text"]

# We print the original report.
print("Original report:")
print(sample_report)

# We print a separator.
print("\n" + "-" * 100 + "\n")

# We print the generated summary.
print("Generated summary:")
print(sample_summary)


## 10. Output explanation

The summary should capture the main points:

- Which channels performed well.
- Which channels underperformed.
- What the marketing team should improve.

For business use, never judge a summary only by whether it sounds fluent. Check whether it is faithful to the original report.


## 11. Build a reusable summarization function

The function will:

1. Accept long marketing text.
2. Run the summarization model.
3. Return the generated summary.

We also allow the user to control summary length.


In [ ]:
# This function summarizes one marketing report.
def summarize_report(report_text, min_length=35, max_length=90):
    # report_text is the long campaign report we want to shorten.
    # min_length and max_length let us control how short or detailed the summary should be.

    # The summarizer returns a list of dictionaries.
    summary_output = summarizer(
        # Long source text.
        report_text,

        # Minimum number of generated tokens.
        min_length=min_length,

        # Maximum number of generated tokens.
        max_length=max_length,

        # False means use a stable decoding strategy instead of random sampling.
        do_sample=False,
    )

    # Extract only the summary text from the first result dictionary.
    summary_text = summary_output[0]["summary_text"]

    # Return the final generated summary.
    return summary_text


In [ ]:
# We create an empty list to store summaries.
generated_summaries = []

# We loop through every campaign report.
for row_index, row in reports_df.iterrows():
    # We summarize the report text.
    summary = summarize_report(row["report_text"])

    # We append campaign information and summary.
    generated_summaries.append(
        {
            "campaign_name": row["campaign_name"],
            "channel_focus": row["channel_focus"],
            "summary": summary,
        }
    )

# We convert summaries into a DataFrame.
summaries_df = pd.DataFrame(generated_summaries)

# We display the summaries.
summaries_df


## 12. Business interpretation

A marketing manager can use these summaries to quickly answer:

- What worked?
- What did not work?
- Which channel needs budget changes?
- What should the team do next?

Now we add a simple business recommendation layer.


In [ ]:
# This function creates a basic recommendation using keywords in the report.
# In a real project, this could be replaced with a stronger model or human review.
def create_marketing_recommendation(report_text):
    # We convert text to lowercase to make keyword checks easier.
    lower_text = report_text.lower()

    # If paid search has weak quality, recommend budget optimization.
    if "broad match" in lower_text or "weak lead quality" in lower_text:
        return "Review paid search targeting and move budget toward high-intent keywords"

    # If approval delays are mentioned, recommend process improvement.
    if "approvals" in lower_text or "legal teams" in lower_text:
        return "Improve customer story approval workflow and prepare more reusable proof assets"

    # If webinar attendance dropped, recommend better scheduling.
    if "attendance dropped" in lower_text or "reminder emails were sent too late" in lower_text:
        return "Improve webinar reminder timing and avoid conflicts with industry events"

    # Otherwise recommend improving call to action.
    return "Simplify campaign call to action and focus each asset on one next step"

# We apply the recommendation function to each report.
reports_df["business_recommendation"] = reports_df["report_text"].apply(create_marketing_recommendation)

# We combine campaign name, summary, and recommendation.
marketing_brief_df = summaries_df.merge(
    reports_df[["campaign_name", "business_recommendation"]],
    on="campaign_name",
)

# We display the business brief.
marketing_brief_df


## 13. Handling longer reports

Transformer models have input limits. If a report is very long, we can split it into chunks, summarize each chunk, and then summarize the combined chunk summaries.

This is called hierarchical summarization.

The simple function below demonstrates the idea for classroom use.


In [ ]:
# This function splits text into chunks based on word count.
def split_text_into_chunks(text, words_per_chunk=120):
    # We split the text into individual words.
    words = text.split()

    # We create an empty list for chunks.
    chunks = []

    # We move through the words in steps of words_per_chunk.
    for start_index in range(0, len(words), words_per_chunk):
        # We select a group of words for one chunk.
        chunk_words = words[start_index:start_index + words_per_chunk]

        # We join the words back into a string.
        chunk_text = " ".join(chunk_words)

        # We add the chunk to the list.
        chunks.append(chunk_text)

    # We return all chunks.
    return chunks

# This function summarizes long text by summarizing chunks first.
def summarize_long_report(report_text):
    # We split the report into chunks.
    chunks = split_text_into_chunks(report_text, words_per_chunk=120)

    # We summarize each chunk.
    chunk_summaries = [
        summarize_report(chunk, min_length=20, max_length=60)
        for chunk in chunks
    ]

    # We combine the chunk summaries into one text.
    combined_summary_text = " ".join(chunk_summaries)

    # If there is only one chunk, we return its summary.
    if len(chunk_summaries) == 1:
        return combined_summary_text

    # If there are multiple chunks, we summarize the combined summaries.
    final_summary = summarize_report(combined_summary_text, min_length=30, max_length=90)

    # We return the final summary.
    return final_summary


In [ ]:
# We test the long-report summarization function.
long_summary = summarize_long_report(reports_df.loc[1, "report_text"])

# We print the summary.
print(long_summary)


## 14. Quality checklist for summaries

A good business summary should be:

- Accurate: it should not invent facts.
- Short: it should remove unnecessary detail.
- Useful: it should help someone decide what to do.
- Balanced: it should mention both strengths and weaknesses.
- Clear: it should use simple language.

For important business decisions, a human should review the summary.


In [ ]:
# We create a simple manual review checklist.
summary_review_checklist = pd.DataFrame(
    [
        {"check": "Does the summary mention the main winning channel?", "status": "To review"},
        {"check": "Does the summary mention the main weak area?", "status": "To review"},
        {"check": "Does the summary avoid unsupported claims?", "status": "To review"},
        {"check": "Is the summary short enough for a manager?", "status": "To review"},
        {"check": "Can the team take action from this summary?", "status": "To review"},
    ]
)

# We display the checklist.
summary_review_checklist


## 15. Limitations and risks

Summarization models are powerful, but they can fail.

Risks:

- The model may omit an important detail.
- The model may overemphasize a minor point.
- The model may generate a statement not supported by the original report.
- Long reports may lose detail during chunking.
- Marketing numbers and dates must be checked carefully.

Use summarization as a first draft, not as an unquestioned final report.


## 16. Alternatives

Other options:

- Manual executive summaries.
- Extractive summarization using sentence ranking.
- T5 or PEGASUS summarization models.
- LLM-based summarization with strict source grounding.
- Dashboard-based reporting with generated commentary.

## Student practice

1. Add one new campaign report.
2. Change `max_length` and observe the summary.
3. Compare short and long summaries.
4. Highlight one sentence that should not be trusted without review.
5. Create a final marketing brief for leadership.

## Final takeaway

You built an end-to-end marketing summarization workflow that turns long campaign reports into short summaries and business recommendations.
